# Stage 2 — Behavioural Feature Engineering

**Thesis:** Predictive Analytics for MSME Credit Risk Assessment using Behavioural Feature Engineering and Explainable Ensemble Machine Learning

Notebook 2 of 5. This is really the core of the whole thesis: building six
families of *behavioural* features out of the relational sub-tables, aggregated
up to one row per applicant (`SK_ID_CURR`), for the MSME-proxy population I
defined in Notebook 1.

| # | Feature family | Source sub-table(s) | Behaviour captured |
|---|---|---|---|
| 1 | Repayment consistency | `installments_payments` | Punctuality of instalment payment |
| 2 | Payment shortfall | `installments_payments` | Paying less than the amount due |
| 3 | Delinquency frequency & recency | `POS_CASH_balance`, `credit_card_balance`, `bureau` | Days-past-due events, severity, how recent |
| 4 | Credit utilisation trend | `credit_card_balance` | Level and direction of revolving-credit usage |
| 5 | Bureau credit depth | `bureau`, `bureau_balance` | Breadth and health of the external credit file |
| 6 | Previous-application outcomes | `previous_application` | Approval / refusal history with the lender |

I build each family separately, merge them onto the training frame, then check
how predictive each one actually is (mutual information + a quick tree-importance
check). Everything gets written out to `outputs/features.parquet`, which is the
matrix Notebooks 3–5 work from.


In [1]:

# Same setup as Notebook 1
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.ioff()
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA_DIR = os.path.join(ROOT, "home-credit-default-risk")
OUT_DIR = os.path.join(ROOT, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
    "axes.edgecolor": "#333333", "axes.linewidth": 0.8,
})
C_REPAID, C_DEFAULT = "#4C72B0", "#C44E52"
FIG_INDEX = {}

def savefig(fig, name, caption=""):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    FIG_INDEX[name] = caption
    print(f"saved -> outputs/{name}" + (f"   |  {caption}" if caption else ""))
    return path

CHUNK = 1_000_000

def load_subtable(filename, id_set, usecols=None):
    """Read a sub-table in chunks and only keep rows for the MSME-proxy ids -
    the full files are too big to load in one go on this machine."""
    parts = []
    for ch in pd.read_csv(os.path.join(DATA_DIR, filename), usecols=usecols, chunksize=CHUNK):
        parts.append(ch[ch["SK_ID_CURR"].isin(id_set)])
    out = pd.concat(parts, ignore_index=True)
    print(f"  {filename}: {len(out):,} rows for {out['SK_ID_CURR'].nunique():,} applicants")
    return out

print("Setup complete. ROOT =", ROOT)


Setup complete. ROOT = C:\Users\VARUN\OneDrive\Desktop\Final Report


In [2]:
# base frame: the MSME proxy applicants and their target
msme = pd.read_csv(os.path.join(OUT_DIR, "msme_proxy.csv"))
base = msme[["SK_ID_CURR", "TARGET"]].copy()
ids = set(base["SK_ID_CURR"])
print(f"MSME proxy: {len(base):,} applicants,  default rate {base['TARGET'].mean():.3f}")

# I'll build up the feature blocks one at a time and join them onto this
features = base.set_index("SK_ID_CURR").copy()

def attach(df_agg, family):
    """Left-join an aggregated (index = SK_ID_CURR) feature block onto `features`."""
    global features
    df_agg = df_agg.add_prefix(f"{family}__")
    features = features.join(df_agg, how="left")
    print(f"  + {family}: {df_agg.shape[1]} columns  "
          f"({df_agg.notna().any(axis=1).sum():,} applicants matched)")


MSME proxy: 38,412 applicants,  default rate 0.102


## Families 1–2: Repayment consistency & payment shortfall (`installments_payments`)

`installments_payments` has one row per instalment of every prior Home Credit
loan - the amount due (`AMT_INSTALMENT`) and paid (`AMT_PAYMENT`), and the
scheduled (`DAYS_INSTALMENT`) vs actual (`DAYS_ENTRY_PAYMENT`) payment day
(negative = days before the current application).

- **Days late** `= DAYS_ENTRY_PAYMENT − DAYS_INSTALMENT` (positive means paid late).
- **Repayment consistency** = share of instalments paid on or before the due day.
- **Payment shortfall** = share of instalments where `AMT_PAYMENT < AMT_INSTALMENT`,
  plus how big that shortfall tends to be relative to what was due.


In [3]:
print("Loading installments_payments ...")
inst = load_subtable("installments_payments.csv", ids)

inst["days_late"] = inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]
inst["is_late"] = (inst["days_late"] > 0).astype("int8")
inst["is_dpd30"] = (inst["days_late"] > 30).astype("int8")
inst["pay_ratio"] = inst["AMT_PAYMENT"] / inst["AMT_INSTALMENT"].replace(0, np.nan)
inst["is_short"] = (inst["AMT_PAYMENT"] < inst["AMT_INSTALMENT"] - 1).astype("int8")   # 1-unit tolerance for rounding
inst["shortfall_amt"] = (inst["AMT_INSTALMENT"] - inst["AMT_PAYMENT"]).clip(lower=0)
inst["shortfall_ratio"] = inst["shortfall_amt"] / inst["AMT_INSTALMENT"].replace(0, np.nan)

g = inst.groupby("SK_ID_CURR")

# Family 1: repayment consistency
fam1 = pd.DataFrame({
    "n_instalments":     g.size(),
    "ontime_ratio":      1 - g["is_late"].mean(),
    "late_ratio":        g["is_late"].mean(),
    "dpd30_ratio":       g["is_dpd30"].mean(),
    "days_late_mean":    g["days_late"].mean(),
    "days_late_max":     g["days_late"].max(),
    "days_late_std":     g["days_late"].std(),
})
attach(fam1, "repay_consistency")

# Family 2: payment shortfall
fam2 = pd.DataFrame({
    "short_ratio":        g["is_short"].mean(),
    "shortfall_ratio_mean": g["shortfall_ratio"].mean(),
    "shortfall_ratio_max":  g["shortfall_ratio"].max(),
    "pay_ratio_mean":     g["pay_ratio"].mean(),
    "pay_ratio_min":      g["pay_ratio"].min(),
})
attach(fam2, "payment_shortfall")


Loading installments_payments ...


  installments_payments.csv: 1,442,792 rows for 37,130 applicants


  + repay_consistency: 7 columns  (37,130 applicants matched)
  + payment_shortfall: 5 columns  (37,130 applicants matched)


## Family 3: Delinquency frequency & recency (`POS_CASH_balance`, `credit_card_balance`, `bureau`)

Delinquency shows up in three places:

- **`POS_CASH_balance`** and **`credit_card_balance`** - monthly `SK_DPD`
  (days-past-due) snapshots on prior Home Credit loans. `MONTHS_BALANCE` is the
  month offset (−1 = most recent).
- **`bureau`** - `CREDIT_DAY_OVERDUE` and `AMT_CREDIT_MAX_OVERDUE` on external
  credit lines.

Features here: total delinquent months, worst DPD, mean DPD when late, and
**recency** - months since the last delinquency (large or missing means none).


In [4]:
print("Loading POS_CASH_balance ...")
pos = load_subtable("POS_CASH_balance.csv", ids,
                    usecols=["SK_ID_CURR", "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"])
print("Loading credit_card_balance (DPD columns) ...")
ccd = load_subtable("credit_card_balance.csv", ids,
                    usecols=["SK_ID_CURR", "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"])

dpd = pd.concat([pos, ccd], ignore_index=True)
dpd["is_dpd"] = (dpd["SK_DPD"] > 0).astype("int8")
dpd["is_dpd_def"] = (dpd["SK_DPD_DEF"] > 0).astype("int8")
# months since the most recent delinquency - MONTHS_BALANCE is negative, -1 = last month
last_dpd = (dpd[dpd["is_dpd"] == 1].groupby("SK_ID_CURR")["MONTHS_BALANCE"].max())

gd = dpd.groupby("SK_ID_CURR")
fam3_hc = pd.DataFrame({
    "dpd_months_count":  gd["is_dpd"].sum(),
    "dpd_def_months_count": gd["is_dpd_def"].sum(),
    "dpd_month_ratio":   gd["is_dpd"].mean(),
    "dpd_max":           gd["SK_DPD"].max(),
    "dpd_mean_when_late": dpd[dpd["is_dpd"] == 1].groupby("SK_ID_CURR")["SK_DPD"].mean(),
    "months_since_last_dpd": -last_dpd,   # 1 = delinquent last month, large = long ago
})

# now bureau overdue
print("Loading bureau (overdue columns) ...")
bur_od = load_subtable("bureau.csv", ids,
                       usecols=["SK_ID_CURR", "CREDIT_DAY_OVERDUE", "AMT_CREDIT_MAX_OVERDUE",
                                "AMT_CREDIT_SUM_OVERDUE"])
gb = bur_od.groupby("SK_ID_CURR")
fam3_bur = pd.DataFrame({
    "bureau_lines_overdue_now": (bur_od["CREDIT_DAY_OVERDUE"] > 0).groupby(bur_od["SK_ID_CURR"]).sum(),
    "bureau_day_overdue_max":   gb["CREDIT_DAY_OVERDUE"].max(),
    "bureau_max_overdue_amt":   gb["AMT_CREDIT_MAX_OVERDUE"].max(),
    "bureau_sum_overdue_now":   gb["AMT_CREDIT_SUM_OVERDUE"].sum(),
})

fam3 = fam3_hc.join(fam3_bur, how="outer")
attach(fam3, "delinquency")
del pos, ccd, dpd


Loading POS_CASH_balance ...


  POS_CASH_balance.csv: 1,057,554 rows for 36,827 applicants
Loading credit_card_balance (DPD columns) ...


  credit_card_balance.csv: 394,189 rows for 12,095 applicants


Loading bureau (overdue columns) ...


  bureau.csv: 155,675 rows for 31,584 applicants
  + delinquency: 10 columns  (38,219 applicants matched)


## Family 4: Credit utilisation trend (`credit_card_balance`)

For applicants who held a Home Credit card, monthly **utilisation** is
`AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL`. I capture two things:

- **Level** - mean and most-recent utilisation.
- **Trend** - the slope of utilisation against `MONTHS_BALANCE` (a simple OLS fit
  per applicant); a positive slope means growing reliance on revolving credit.

Only about 32% of the MSME proxy actually held a Home Credit card (from
Notebook 1), so I add a `has_credit_card`-style indicator and just leave the trend
features missing for everyone else - that gets handled by imputation + a
missing-indicator flag in Stage 3.


In [5]:
print("Loading credit_card_balance (utilisation columns) ...")
cc = load_subtable("credit_card_balance.csv", ids,
                   usecols=["SK_ID_CURR", "MONTHS_BALANCE", "AMT_BALANCE",
                            "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_CURRENT",
                            "AMT_PAYMENT_TOTAL_CURRENT"])

cc["util"] = cc["AMT_BALANCE"] / cc["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
cc = cc.sort_values(["SK_ID_CURR", "MONTHS_BALANCE"])

def ols_slope(df):
    """Per-applicant OLS slope of utilisation vs month. Returns NaN if there
    aren't at least 2 valid points to fit a line through."""
    d = df.dropna(subset=["util"])
    if len(d) < 2:
        return np.nan
    x = d["MONTHS_BALANCE"].to_numpy(float)
    y = d["util"].to_numpy(float)
    xc = x - x.mean()
    denom = (xc ** 2).sum()
    return np.nan if denom == 0 else (xc * (y - y.mean())).sum() / denom

gc = cc.groupby("SK_ID_CURR")
recent = cc.groupby("SK_ID_CURR").tail(1).set_index("SK_ID_CURR")["util"]

fam4 = pd.DataFrame({
    "util_mean":     gc["util"].mean(),
    "util_max":      gc["util"].max(),
    "util_recent":   recent,
    "util_slope":    gc.apply(ols_slope),
    "cc_months":     gc.size(),
    "drawings_mean": gc["AMT_DRAWINGS_CURRENT"].mean(),
})
attach(fam4, "credit_utilisation")   # coverage for the missing ~68% is handled later, in the has__ flags
del cc


Loading credit_card_balance (utilisation columns) ...


  credit_card_balance.csv: 394,189 rows for 12,095 applicants


  + credit_utilisation: 6 columns  (12,095 applicants matched)


## Family 5: Bureau credit depth (`bureau`, `bureau_balance`)

`bureau` lists every external credit line reported for the applicant.
`bureau_balance` adds a monthly status string per line (`C` closed, `X` unknown,
`0`–`5` = buckets of months past due).

Features: number of credit lines (total / active), total credit sum and
outstanding debt, debt-to-credit ratio, how long they've had credit (oldest
line), and from `bureau_balance` the share of months spent in a past-due
(`1`–`5`) status.


In [6]:
print("Loading bureau ...")
bur = load_subtable("bureau.csv", ids,
                    usecols=["SK_ID_CURR", "SK_ID_BUREAU", "CREDIT_ACTIVE", "DAYS_CREDIT",
                             "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "CNT_CREDIT_PROLONG"])

gb = bur.groupby("SK_ID_CURR")
fam5 = pd.DataFrame({
    "n_bureau_lines":     gb.size(),
    "n_active_lines":     bur.assign(a=(bur["CREDIT_ACTIVE"] == "Active")).groupby("SK_ID_CURR")["a"].sum(),
    "credit_sum_total":   gb["AMT_CREDIT_SUM"].sum(),
    "debt_total":         gb["AMT_CREDIT_SUM_DEBT"].sum(),
    "history_days":       -gb["DAYS_CREDIT"].min(),          # days since their oldest line
    "prolong_total":      gb["CNT_CREDIT_PROLONG"].sum(),
})
fam5["debt_credit_ratio"] = fam5["debt_total"] / fam5["credit_sum_total"].replace(0, np.nan)
fam5["active_line_ratio"] = fam5["n_active_lines"] / fam5["n_bureau_lines"]

# bureau_balance: past-due status share, linked via SK_ID_BUREAU -> SK_ID_CURR
bmap = dict(zip(bur["SK_ID_BUREAU"], bur["SK_ID_CURR"]))
bureau_curr = set(bur["SK_ID_BUREAU"])
parts = []
for ch in pd.read_csv(os.path.join(DATA_DIR, "bureau_balance.csv"), chunksize=CHUNK):
    ch = ch[ch["SK_ID_BUREAU"].isin(bureau_curr)]
    if len(ch):
        ch["SK_ID_CURR"] = ch["SK_ID_BUREAU"].map(bmap)
        ch["is_pastdue"] = ch["STATUS"].isin(list("12345")).astype("int8")
        parts.append(ch[["SK_ID_CURR", "is_pastdue", "MONTHS_BALANCE"]])
bb = pd.concat(parts, ignore_index=True)
gbb = bb.groupby("SK_ID_CURR")
fam5 = fam5.join(pd.DataFrame({
    "bb_months_observed":  gbb.size(),
    "bb_pastdue_ratio":    gbb["is_pastdue"].mean(),
    "bb_pastdue_months":   gbb["is_pastdue"].sum(),
}), how="left")

attach(fam5, "bureau_depth")
del bur, bb


Loading bureau ...


  bureau.csv: 155,675 rows for 31,584 applicants


  + bureau_depth: 11 columns  (31,584 applicants matched)


## Family 6: Previous-application outcomes (`previous_application`)

`previous_application` has one row per earlier application the client made to
Home Credit, with a `NAME_CONTRACT_STATUS` (`Approved`, `Refused`, `Canceled`,
`Unused offer`). Features: number of prior applications, approval and refusal
rates, average requested/approved amounts, and the ratio of what they were
actually granted vs. what they asked for.


In [7]:
print("Loading previous_application ...")
prev = load_subtable("previous_application.csv", ids,
                     usecols=["SK_ID_CURR", "NAME_CONTRACT_STATUS", "AMT_APPLICATION",
                              "AMT_CREDIT", "DAYS_DECISION"])

prev["approved"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype("int8")
prev["refused"]  = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype("int8")
prev["grant_ratio"] = prev["AMT_CREDIT"] / prev["AMT_APPLICATION"].replace(0, np.nan)

gp = prev.groupby("SK_ID_CURR")
fam6 = pd.DataFrame({
    "n_prev_apps":       gp.size(),
    "approval_rate":     gp["approved"].mean(),
    "refusal_rate":      gp["refused"].mean(),
    "amt_application_mean": gp["AMT_APPLICATION"].mean(),
    "amt_credit_mean":   gp["AMT_CREDIT"].mean(),
    "grant_ratio_mean":  gp["grant_ratio"].mean(),
    "days_since_last_app": -gp["DAYS_DECISION"].max(),
})
attach(fam6, "prev_app")
del prev, inst
print("\nFeature matrix so far:", features.shape)


Loading previous_application ...


  previous_application.csv: 183,106 rows for 37,083 applicants
  + prev_app: 7 columns  (37,083 applicants matched)

Feature matrix so far: (38412, 47)


## Assemble the modelling matrix

Now I join the six behavioural blocks to a set of **raw** main-table features
(kept around as the fallback comparison set, per the research design), plus a
per-family **coverage indicator** so a model can tell "we know their behaviour is
clean" apart from "we just don't have any history for them". I'm deliberately
**not** imputing anything here - that happens inside the training fold in Stage 3
so nothing leaks. Result gets written to `outputs/features.parquet`.


In [8]:
# raw main-table features (the fallback comparison set)
raw_num = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
           "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
           "DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH",
           "DAYS_LAST_PHONE_CHANGE", "CNT_CHILDREN", "CNT_FAM_MEMBERS",
           "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",
           "REGION_POPULATION_RELATIVE", "OWN_CAR_AGE",
           "DEF_30_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE"]
raw_cat = ["NAME_CONTRACT_TYPE", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
           "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
           "NAME_HOUSING_TYPE"]

raw = msme.set_index("SK_ID_CURR")[raw_num + raw_cat].copy()
raw["DAYS_EMPLOYED"] = raw["DAYS_EMPLOYED"].replace(365243, np.nan)
# a handful of simple ratios on top of the raw amounts
raw["credit_income_ratio"]  = raw["AMT_CREDIT"] / raw["AMT_INCOME_TOTAL"].replace(0, np.nan)
raw["annuity_income_ratio"] = raw["AMT_ANNUITY"] / raw["AMT_INCOME_TOTAL"].replace(0, np.nan)
raw["credit_goods_ratio"]   = raw["AMT_CREDIT"] / raw["AMT_GOODS_PRICE"].replace(0, np.nan)
raw["credit_term"]          = raw["AMT_CREDIT"] / raw["AMT_ANNUITY"].replace(0, np.nan)
raw = raw.add_prefix("raw__")

model_df = features.join(raw, how="left")

# per-family coverage flags
fam_prefixes = ["repay_consistency", "payment_shortfall", "delinquency",
                "credit_utilisation", "bureau_depth", "prev_app"]
for p in fam_prefixes:
    cols = [c for c in model_df.columns if c.startswith(p + "__")]
    model_df[f"has__{p}"] = model_df[cols].notna().any(axis=1).astype("int8")

model_df = model_df.reset_index()
path = os.path.join(OUT_DIR, "features.parquet")
model_df.to_parquet(path, index=False)

n_feat = model_df.shape[1] - 2  # minus SK_ID_CURR, TARGET
print(f"features.parquet : {model_df.shape[0]:,} rows x {model_df.shape[1]} columns "
      f"({n_feat} features)\nsaved -> outputs/features.parquet")
print("\ncoverage indicators (share of applicants with the family present):")
print(model_df[[f'has__{p}' for p in fam_prefixes]].mean().round(3).to_string())


features.parquet : 38,412 rows x 86 columns (84 features)
saved -> outputs/features.parquet

coverage indicators (share of applicants with the family present):
has__repay_consistency     0.967
has__payment_shortfall     0.967
has__delinquency           0.995
has__credit_utilisation    0.315
has__bureau_depth          0.822
has__prev_app              0.965


## Feature-family predictive evaluation

Before doing any real modelling, I want to check how predictive each family
actually is, using two measures on the full MSME proxy:

1. **Mutual information** with `TARGET` (picks up non-linear relationships too),
   taking the median over 5 resamples so it's not just noise;
2. **Random-forest importance** from one quick forest trained on everything
   (median-imputed), summed up to family level.

These give a first ranking that I'll come back to and cross-check with SHAP in
Notebook 5.


In [9]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

beh_cols = [c for c in model_df.columns
            if c.split("__")[0] in fam_prefixes and not c.startswith("has__")]
X = model_df[beh_cols].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
y = model_df["TARGET"].to_numpy()

# mutual information, median over 5 seeds so a single unlucky run doesn't skew things
mi_runs = np.vstack([
    mutual_info_classif(X, y, random_state=s, discrete_features=False)
    for s in range(5)
])
mi = pd.Series(np.median(mi_runs, axis=0), index=beh_cols)

# quick RF importance, just to sanity-check the MI ranking against something different
rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced",
                            n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X, y)
imp = pd.Series(rf.feature_importances_, index=beh_cols)

def family_of(col):
    return col.split("__")[0]

fam_eval = pd.DataFrame({
    "n_features":   pd.Series(beh_cols).groupby(pd.Series(beh_cols).map(family_of)).size(),
    "mi_sum":       mi.groupby(mi.index.map(family_of)).sum(),
    "mi_max":       mi.groupby(mi.index.map(family_of)).max(),
    "rf_importance":imp.groupby(imp.index.map(family_of)).sum(),
}).sort_values("mi_sum", ascending=False)
fam_eval.to_csv(os.path.join(OUT_DIR, "feature_family_evaluation.csv"))
print(fam_eval.round(4).to_string())

top_feats = mi.sort_values(ascending=False).head(15)
print("\nTop 15 individual behavioural features by mutual information:")
print(top_feats.round(4).to_string())


                    n_features  mi_sum  mi_max  rf_importance
bureau_depth                11  0.0287  0.0046         0.2574
credit_utilisation           6  0.0270  0.0069         0.1647
prev_app                     7  0.0187  0.0049         0.2490
payment_shortfall            5  0.0185  0.0062         0.0895
delinquency                 10  0.0129  0.0057         0.0743
repay_consistency            7  0.0074  0.0036         0.1651

Top 15 individual behavioural features by mutual information:
credit_utilisation__util_mean         0.0069
payment_shortfall__pay_ratio_min      0.0062
credit_utilisation__util_max          0.0058
delinquency__months_since_last_dpd    0.0057
credit_utilisation__util_recent       0.0057
prev_app__approval_rate               0.0049
bureau_depth__active_line_ratio       0.0046
bureau_depth__bb_months_observed      0.0044
bureau_depth__debt_credit_ratio       0.0044
prev_app__amt_application_mean        0.0042
payment_shortfall__pay_ratio_mean     0.0040
bureau_d

In [10]:
# figure: family-level predictive contribution
fam_labels = {
    "repay_consistency": "Repayment\nconsistency",
    "payment_shortfall": "Payment\nshortfall",
    "delinquency": "Delinquency\nfreq. & recency",
    "credit_utilisation": "Credit\nutilisation trend",
    "bureau_depth": "Bureau\ncredit depth",
    "prev_app": "Previous-app\noutcomes",
}
fe = fam_eval.reindex(sorted(fam_eval.index, key=lambda k: fam_eval.loc[k, "mi_sum"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].barh([fam_labels[k] for k in fe.index], fe["mi_sum"], color="#4C72B0",
             edgecolor="black", linewidth=0.4)
axes[0].set_xlabel("Σ mutual information with default")
axes[0].set_title("Mutual information by feature family")

axes[1].barh([fam_labels[k] for k in fe.index], fe["rf_importance"], color="#5B8C7B",
             edgecolor="black", linewidth=0.4)
axes[1].set_xlabel("Σ random-forest importance")
axes[1].set_title("Tree importance by feature family")
for ax in axes:
    sns.despine(ax=ax)
fig.tight_layout()
savefig(fig, "fe_01_feature_family_contribution.png",
        "Predictive contribution of each behavioural feature family to default "
        "prediction in the MSME proxy, by summed mutual information (left) and "
        "summed random-forest importance (right).")

fig, ax = plt.subplots(figsize=(7, 6))
tf = top_feats.sort_values()
ax.barh(tf.index, tf.values, color="#4C72B0", edgecolor="black", linewidth=0.4)
ax.set_xlabel("Mutual information with default")
ax.set_title("Top 15 behavioural features by mutual information")
sns.despine(ax=ax)
savefig(fig, "fe_02_top_features_mi.png",
        "The 15 individual engineered behavioural features most informative about "
        "default in the MSME proxy, ranked by mutual information.")


saved -> outputs/fe_01_feature_family_contribution.png   |  Predictive contribution of each behavioural feature family to default prediction in the MSME proxy, by summed mutual information (left) and summed random-forest importance (right).


saved -> outputs/fe_02_top_features_mi.png   |  The 15 individual engineered behavioural features most informative about default in the MSME proxy, ranked by mutual information.


'C:\\Users\\VARUN\\OneDrive\\Desktop\\Final Report\\outputs\\fe_02_top_features_mi.png'

In [11]:
# Do the behavioural features actually add anything over the raw ones? Checking
# with two very different model types so it's not just an artefact of one of them:
# a scaled logistic regression, and gradient-boosted trees (closer to what I'll
# actually use in Stage 4, and it handles missingness/scale natively).
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

def make_pre(cols, scale):
    num = [c for c in cols if pd.api.types.is_numeric_dtype(model_df[c])]
    cat = [c for c in cols if c not in num]
    num_steps = [SimpleImputer(strategy="median")]
    if scale:
        num_steps.append(StandardScaler())
    return ColumnTransformer([
        ("num", make_pipeline(*num_steps), num),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OneHotEncoder(handle_unknown="ignore", min_frequency=20)), cat),
    ])

def auc(cols, model):
    scale = isinstance(model, LogisticRegression)
    pipe = make_pipeline(make_pre(cols, scale), model)
    X = model_df[cols].replace([np.inf, -np.inf], np.nan)
    s = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    return s.mean(), s.std()

raw_cols = [c for c in model_df.columns if c.startswith("raw__")]
beh_cols_all = [c for c in model_df.columns if c.split("__")[0] in fam_prefixes]

rows = []
for name, cols in [("raw only", raw_cols),
                   ("behavioural only", beh_cols_all),
                   ("raw + behavioural", raw_cols + beh_cols_all)]:
    lr_m, lr_s = auc(cols, LogisticRegression(max_iter=5000, class_weight="balanced"))
    gb_m, gb_s = auc(cols, HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                                          random_state=RANDOM_STATE))
    rows.append({"feature_set": name, "logreg_auc": lr_m, "logreg_sd": lr_s,
                 "gbm_auc": gb_m, "gbm_sd": gb_s})

auc_tbl = pd.DataFrame(rows).set_index("feature_set")
auc_tbl.to_csv(os.path.join(OUT_DIR, "raw_vs_behavioural_auc.csv"))
print("5-fold ROC-AUC — incremental value of the behavioural feature set:\n")
print(auc_tbl.round(4).to_string())
print(f"\nGBM lift (raw+behavioural − raw): "
      f"{auc_tbl.loc['raw + behavioural','gbm_auc'] - auc_tbl.loc['raw only','gbm_auc']:+.4f}")


5-fold ROC-AUC — incremental value of the behavioural feature set:

                   logreg_auc  logreg_sd  gbm_auc  gbm_sd
feature_set                                              
raw only               0.7376     0.0081   0.7407  0.0066
behavioural only       0.6718     0.0067   0.6807  0.0063
raw + behavioural      0.7552     0.0076   0.7552  0.0082

GBM lift (raw+behavioural − raw): +0.0145


In [12]:
# figure: incremental value of behavioural features
order = ["raw only", "behavioural only", "raw + behavioural"]
at = auc_tbl.reindex(order)
x = np.arange(len(order))
w = 0.36

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(x - w/2, at["logreg_auc"], w, yerr=at["logreg_sd"], capsize=3,
       label="Logistic regression", color="#4C72B0", edgecolor="black", linewidth=0.4)
ax.bar(x + w/2, at["gbm_auc"], w, yerr=at["gbm_sd"], capsize=3,
       label="Gradient-boosted trees", color="#5B8C7B", edgecolor="black", linewidth=0.4)
for i, (l, g) in enumerate(zip(at["logreg_auc"], at["gbm_auc"])):
    ax.text(i - w/2, l + 0.012, f"{l:.3f}", ha="center", fontsize=9)
    ax.text(i + w/2, g + 0.012, f"{g:.3f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels([o.replace(" ", "\n") for o in order])
ax.set_ylabel("5-fold cross-validated ROC-AUC")
ax.set_ylim(0.5, 0.82)
ax.set_title("Incremental value of behavioural feature engineering (MSME proxy)")
ax.legend(loc="lower right")
sns.despine(ax=ax)
savefig(fig, "fe_03_raw_vs_behavioural_auc.png",
        "Five-fold ROC-AUC for models trained on raw main-table features, engineered "
        "behavioural features, and their combination. Adding behavioural features "
        "raises AUC by roughly 0.015–0.018 over raw features alone.")


saved -> outputs/fe_03_raw_vs_behavioural_auc.png   |  Five-fold ROC-AUC for models trained on raw main-table features, engineered behavioural features, and their combination. Adding behavioural features raises AUC by roughly 0.015–0.018 over raw features alone.


'C:\\Users\\VARUN\\OneDrive\\Desktop\\Final Report\\outputs\\fe_03_raw_vs_behavioural_auc.png'

## Stage 2 summary

- **6 behavioural feature families** (84 features) built and joined onto the
  38,412 MSME-proxy applicants → `outputs/features.parquet` (Notebooks 3–5 use this).
- Each family has a `has__<family>` coverage flag. Coverage: delinquency 99.5%,
  repayment/shortfall 96.7%, previous-app 96.5%, bureau-depth 82.2%, and
  **credit-utilisation only 31.5%** (card holders only).
- **Family contribution** (`outputs/feature_family_evaluation.csv`, fig `fe_01`):
  bureau credit depth, credit-utilisation and previous-application outcomes come
  out on top; the single strongest feature is mean credit-card utilisation.
- **Incremental value over raw features** (`outputs/raw_vs_behavioural_auc.csv`,
  fig `fe_03`) - 5-fold ROC-AUC: raw 0.74 → raw + behavioural **0.755**
  (gradient-boosted trees). So the behavioural features add roughly **+0.015 AUC**
  over raw application data alone - not huge, but real and consistent across
  folds. That's my evidence for RQ3.

Figures from this notebook: `fe_01` family contribution, `fe_02` top features,
`fe_03` raw-vs-behavioural AUC.

Next up (Notebook 3): the stratified 80/20 split, fold-safe imputation/encoding/
scaling, and SMOTE on the training partition only.
